# 10 modelos 22 a 24 y conclusiones



### Modelo 22 — Revisión de préstamos pequeños

**Qué probamos.** Dividimos las solicitudes en grupos según el monto y comparamos entrenamiento, validación y test. También revisamos probabilidades de pago y defaults por monto y tasa de interés.

**Qué pasó.** Encontramos pocos préstamos pequeños en los datos históricos. Por ejemplo, hasta USD 1.200 representaban aproximadamente el 1,2% del entrenamiento reducido, pero el 36,8% del test. Los ingresos para montos comparables también eran mayores en enero. En los aprobados históricos hubo 23 defaults reales frente a 20,53 esperados: no apareció un error grande de calibración general.

**Qué decidimos.** Utilizar el análisis para entender las limitaciones de la validación, sin cambiar automáticamente el modelo. La diferencia entre poblaciones no alcanza por sí sola para demostrar qué préstamos de enero pagarán.

In [ ]:

import numpy as np
import pandas as pd

# definimos los cinco rangos de monto a partir del test
m22_cortes = np.unique(
    np.quantile(
        m19_test["Monto_Solicitado"],
        [0.20, 0.40, 0.60, 0.80]
    )
)

m22_etiquetas = [
    f"G{i+1}" for i in range(len(m22_cortes) + 1)
]

def asignar_grupos_m22(datos):
    resultado = datos.copy()

    codigos = np.searchsorted(
        m22_cortes,
        resultado["Monto_Solicitado"].to_numpy(float),
        side="right"
    )

    resultado["Grupo_Monto"] = pd.Categorical.from_codes(
        codigos,
        categories=m22_etiquetas,
        ordered=True
    )

    return resultado


# recuperamos las predicciones temporales de la V7
m22_validaciones = []

for mes in m21_meses:
    datos = m21_predicciones[mes]["Datos"].copy()

    probabilidades = m21_predicciones[
        mes
    ]["Probabilidades"]["V7"]

    assert len(datos) == len(probabilidades)

    datos["Prob_Pago"] = probabilidades
    datos["Mes_Validacion"] = mes

    m22_validaciones.append(datos)

m22_val = asignar_grupos_m22(
    pd.concat(m22_validaciones, ignore_index=True)
)

m22_train = asignar_grupos_m22(m19_train)
m22_test = asignar_grupos_m22(m19_test)

m22_train_reducido = m22_train[
    m22_train["Monto_Solicitado"] <= m19_max_test
].copy()


# comparamos la distribucion y las caracteristicas de cada grupo
def resumen_montos_m22(datos, origen):
    tabla = datos.groupby(
        "Grupo_Monto", observed=False
    ).agg(
        Solicitudes=("ID_Solicitud", "size"),
        Monto_Mediano=("Monto_Solicitado", "median"),
        Tasa_Promedio=("Tasa_Interes", "mean"),
        Ingreso_Mediano=("Ingreso_Mensual", "median"),
        Historial_Promedio=("Meses_Observados", "mean")
    ).reset_index()

    tabla["Porcentaje_%"] = (
        100 * tabla["Solicitudes"] / len(datos)
    )

    tabla.insert(0, "Origen", origen)

    return tabla


m22_distribucion = pd.concat([
    resumen_montos_m22(m22_train, "Train completo"),
    resumen_montos_m22(m22_train_reducido, "Train reducido"),
    resumen_montos_m22(m22_val, "Validacion temporal"),
    resumen_montos_m22(m22_test, "Test enero")
], ignore_index=True)

print("DISTRIBUCION DE MONTOS")
display(m22_distribucion.round(3))

print("\nCANTIDAD DE SOLICITUDES POR GRUPO")
display(
    m22_distribucion.pivot(
        index="Grupo_Monto",
        columns="Origen",
        values="Solicitudes"
    )
)

print("\nRANGOS DE MONTO")
print("Cortes:", m22_cortes)
print("Total validaciones:", len(m22_val))


In [ ]:

# calculamos pagos reales y defaults esperados
m22_val["Default_Real"] = 1 - m22_val["Estado_Final"]
m22_val["Default_Esperado"] = 1 - m22_val["Prob_Pago"]


# calculamos la calibracion por grupo de monto
def calibracion_m22(datos, columna):
    tabla = datos.groupby(
        columna, observed=False
    ).agg(
        Solicitudes=("ID_Solicitud", "size"),
        Pago_Real=("Estado_Final", "mean"),
        Pago_Predicho=("Prob_Pago", "mean"),
        Defaults_Reales=("Default_Real", "sum"),
        Defaults_Esperados=("Default_Esperado", "sum")
    )

    tabla["Brecha_pp"] = (
        tabla["Pago_Predicho"] -
        tabla["Pago_Real"]
    ) * 100

    return tabla


print("CALIBRACION POR MONTO")
display(
    calibracion_m22(
        m22_val, "Grupo_Monto"
    ).round(3)
)


# dividimos las tasas de interes en tres grupos
m22_cortes_tasa = np.unique(
    np.quantile(
        m22_test["Tasa_Interes"],
        [1/3, 2/3]
    )
)

m22_val["Grupo_Tasa"] = np.searchsorted(
    m22_cortes_tasa,
    m22_val["Tasa_Interes"].to_numpy(float),
    side="right"
) + 1

print("\nCALIBRACION POR TASA DE INTERES")
display(
    calibracion_m22(
        m22_val, "Grupo_Tasa"
    ).round(3)
)


# revisamos si el problema aparece en los meses mas recientes
print("\nCALIBRACION POR MES")
display(
    calibracion_m22(
        m22_val, "Mes_Validacion"
    ).round(3)
)

print("\nINTERPRETACION")
print("Brecha_pp positiva: el modelo sobreestima el pago.")
print("Brecha_pp negativa: el modelo subestima el pago.")
print("Con pocas solicitudes, las diferencias pueden ser aleatorias.")


In [ ]:

# entrenamos la misma configuracion de la submission v7
m22_modelo = crear_logistica_m19(
    m19_alternativas, C=0.1
)

m22_modelo.fit(
    m22_train_reducido[m19_alternativas],
    m22_train_reducido["Estado_Final"]
)

clases = list(
    m22_modelo.named_steps["modelo"].classes_
)

m22_test["Prob_Pago"] = m22_modelo.predict_proba(
    m22_test[m19_alternativas]
)[:, clases.index(1)]


# calculamos el valor esperado y seleccionamos prestamos
m22_test["EV_Dolar"] = (
    m22_test["Prob_Pago"] *
    (1 + m22_test["Tasa_Interes"]) - 1
)

m22_test["Aprobar"] = 0

ordenados = m22_test.sort_values(
    ["EV_Dolar", "ID_Solicitud"],
    ascending=[False, True]
)

capital = 0.0

for fila in ordenados.itertuples():
    if fila.EV_Dolar <= 0:
        break

    if capital + fila.Monto_Solicitado <= 2_000_000:
        m22_test.at[fila.Index, "Aprobar"] = 1
        capital += fila.Monto_Solicitado

assert capital <= 2_000_000

m22_aprobados = m22_test[
    m22_test["Aprobar"] == 1
].copy()


# resumimos las decisiones por grupo de monto
filas_resumen = []

for grupo in m22_etiquetas:
    candidatos = m22_test[
        m22_test["Grupo_Monto"] == grupo
    ]

    aprobados = candidatos[
        candidatos["Aprobar"] == 1
    ]

    filas_resumen.append({
        "Grupo": grupo,
        "Candidatos": len(candidatos),
        "Aprobados": len(aprobados),
        "Porcentaje_Aprobado": (
            100 * len(aprobados) / len(candidatos)
            if len(candidatos) else 0
        ),
        "Capital": aprobados["Monto_Solicitado"].sum(),
        "Prob_Pago_Media": aprobados["Prob_Pago"].mean(),
        "Tasa_Media": aprobados["Tasa_Interes"].mean(),
        "Defaults_Esperados": (
            1 - aprobados["Prob_Pago"]
        ).sum(),
        "Ganancia_Esperada": (
            aprobados["EV_Dolar"] *
            aprobados["Monto_Solicitado"]
        ).sum()
    })

m22_resumen_test = pd.DataFrame(filas_resumen)

print("CARTERA V7 EN ENERO DE 2026")
display(m22_resumen_test.round(3))


# comparamos las probabilidades entre validacion y test
m22_comparacion_prob = pd.DataFrame({
    "P_Pago_Validacion": m22_val.groupby(
        "Grupo_Monto", observed=False
    )["Prob_Pago"].mean(),

    "P_Pago_Real_Validacion": m22_val.groupby(
        "Grupo_Monto", observed=False
    )["Estado_Final"].mean(),

    "P_Pago_Test": m22_test.groupby(
        "Grupo_Monto", observed=False
    )["Prob_Pago"].mean()
})

m22_comparacion_prob["Cambio_Prob_pp"] = (
    m22_comparacion_prob["P_Pago_Test"] -
    m22_comparacion_prob["P_Pago_Validacion"]
) * 100

print("\nCOMPARACION DE PROBABILIDADES")
display(m22_comparacion_prob.round(3))


# comprobamos si reconstruimos la misma submission
if "cartera_v7" in globals():
    control = m22_test[
        ["ID_Solicitud", "Aprobar"]
    ].merge(
        cartera_v7[
            ["ID_Solicitud", "Decision_Aprobacion"]
        ],
        on="ID_Solicitud",
        validate="one_to_one"
    )

    diferencias = (
        control["Aprobar"] !=
        control["Decision_Aprobacion"]
    ).sum()

    print("\nCONTROL CONTRA SUBMISSION V7")
    print("Decisiones diferentes:", int(diferencias))

    if diferencias > 0:
        print("ATENCION: revisar la reconstruccion antes de concluir.")


print("\nRESUMEN FINAL")
print("Solicitudes aprobadas:", len(m22_aprobados))
print("Capital aprobado:", round(capital, 2))
print("Capital restante:", round(2_000_000 - capital, 2))
print(
    "Defaults esperados:",
    round((1 - m22_aprobados["Prob_Pago"]).sum(), 2)
)
print(
    "Ganancia total esperada:",
    round(
        (
            m22_aprobados["EV_Dolar"] *
            m22_aprobados["Monto_Solicitado"]
        ).sum(), 2
    )
)


### Modelo 23 — Prueba sin variables de ingreso

**Qué probamos.** Quitamos `Ingreso_Mensual`, `Ratio_Monto_Ingreso` y `Saldo_Sobre_Ingreso` de la V7, sin cambiar el resto del entrenamiento ni la selección.

**Qué pasó.** La ganancia mensual promedio subió de USD 281.024,23 a USD 282.068,96, pero la variante solo ganó en 4 de los 7 meses. En diciembre perdió aproximadamente USD 9.606. La probabilidad promedio estimada en test bajó de 89,53% a 88,38%.

**Qué decidimos.** No enviar esta versión. Los resultados no mostraron una mejora estable y tampoco teníamos evidencia de que bajar las probabilidades las hiciera más correctas.

In [ ]:

from sklearn.metrics import roc_auc_score
import pandas as pd
import numpy as np

# mantenemos la v7 y retiramos las variables vinculadas al ingreso
excluir_m23 = {
    "Ingreso_Mensual",
    "Ratio_Monto_Ingreso",
    "Saldo_Sobre_Ingreso"
}

columnas_m23 = [
    c for c in m19_alternativas
    if c not in excluir_m23
]

resultados_m23 = []

# evaluamos los mismos siete meses que en el modelo 21
for mes in m21_meses:
    inicio = pd.Timestamp(mes + "-01")

    entrenamiento = m19_train[
        (m19_train["Fecha_dt"] < inicio) &
        (m19_train["Monto_Solicitado"] <= m19_max_test)
    ]

    validacion = m21_predicciones[mes]["Datos"].copy()

    modelo = crear_logistica_m19(
        columnas_m23, C=0.1
    )

    modelo.fit(
        entrenamiento[columnas_m23],
        entrenamiento["Estado_Final"]
    )

    clases = list(
        modelo.named_steps["modelo"].classes_
    )

    probabilidades = modelo.predict_proba(
        validacion[columnas_m23]
    )[:, clases.index(1)]

    presupuesto = (
        validacion["Monto_Solicitado"].sum() *
        m19_proporcion
    )

    metricas = evaluar_cartera_m21(
        validacion, probabilidades, presupuesto
    )

    base = m21_tabla[
        (m21_tabla["Mes"] == mes) &
        (m21_tabla["Version"] == "V7")
    ].iloc[0]

    resultados_m23.append({
        "Mes": mes,
        "Ganancia_V7": base["Ganancia"],
        "Ganancia_M23": metricas["Ganancia"],
        "Diferencia": metricas["Ganancia"] - base["Ganancia"],
        "ROI_V7": base["ROI_%"],
        "ROI_M23": metricas["ROI_%"],
        "AUC_M23": roc_auc_score(
            validacion["Estado_Final"],
            probabilidades
        ) if validacion["Estado_Final"].nunique() > 1 else np.nan,
        "Defaults_M23": metricas["Defaults"]
    })

    print("Terminado:", mes)

m23_tabla = pd.DataFrame(resultados_m23)

print("\nGANANCIAS POR MES")
display(m23_tabla.round(2))

print("\nRESUMEN")
print("V7:", round(m23_tabla["Ganancia_V7"].mean(), 2))
print("M23:", round(m23_tabla["Ganancia_M23"].mean(), 2))
print("Diferencia promedio:", round(m23_tabla["Diferencia"].mean(), 2))
print("Meses ganados:", int((m23_tabla["Diferencia"] > 0).sum()), "de 7")

# entrenamos la variante final para observar sus probabilidades en test
train_m23 = m19_train[
    m19_train["Monto_Solicitado"] <= m19_max_test
]

modelo_final_m23 = crear_logistica_m19(
    columnas_m23, C=0.1
)

modelo_final_m23.fit(
    train_m23[columnas_m23],
    train_m23["Estado_Final"]
)

clases = list(
    modelo_final_m23.named_steps["modelo"].classes_
)

prob_test_m23 = modelo_final_m23.predict_proba(
    m19_test[columnas_m23]
)[:, clases.index(1)]

comparacion_prob_m23 = m22_test[
    ["ID_Solicitud", "Grupo_Monto", "Prob_Pago"]
].copy()

comparacion_prob_m23["Prob_M23"] = prob_test_m23

print("\nPROBABILIDADES EN TEST POR MONTO")
display(
    comparacion_prob_m23.groupby(
        "Grupo_Monto", observed=False
    )[["Prob_Pago", "Prob_M23"]].mean().round(4)
)

print("\nProbabilidad media V7:", round(
    comparacion_prob_m23["Prob_Pago"].mean(), 4
))
print("Probabilidad media M23:", round(
    comparacion_prob_m23["Prob_M23"].mean(), 4
))


In [ ]:

import pandas as pd
import numpy as np

# recuperamos los prestamos que la V7 habria aprobado historicamente
aprobados_historicos = []

for mes in m21_meses:
    datos = m21_predicciones[mes]["Datos"].reset_index(drop=True)
    probabilidades = m21_predicciones[mes]["Probabilidades"]["V7"]

    presupuesto = datos["Monto_Solicitado"].sum() * m19_proporcion

    cartera = seleccionar_cartera_m20(
        datos, probabilidades, presupuesto
    )

    aprobados = cartera[cartera["Aprobar"] == 1].merge(
        datos[
            [
                "ID_Solicitud", "Estado_Final",
                "Ingreso_Mensual", "Ratio_Monto_Ingreso",
                "Meses_Observados", "Atraso_Maximo"
            ]
        ],
        on="ID_Solicitud",
        validate="one_to_one"
    )

    aprobados["Mes"] = mes
    aprobados_historicos.append(aprobados)

historicos = pd.concat(aprobados_historicos, ignore_index=True)
enero = m22_test[m22_test["Aprobar"] == 1].copy()

# comparamos caracteristicas de las carteras aprobadas
variables = [
    "Monto_Solicitado",
    "Tasa_Interes",
    "Ingreso_Mensual",
    "Ratio_Monto_Ingreso",
    "Meses_Observados",
    "Atraso_Maximo",
    "Prob_Pago"
]

comparacion = pd.DataFrame({
    "Mediana_Historica": historicos[variables].median(),
    "Mediana_Enero": enero[variables].median(),
    "P90_Historico": historicos[variables].quantile(0.90),
    "P10_Historico": historicos[variables].quantile(0.10)
})

comparacion["Test_Sobre_P90_%"] = [
    100 * (enero[c] > comparacion.loc[c, "P90_Historico"]).mean()
    for c in variables
]

comparacion["Test_Bajo_P10_%"] = [
    100 * (enero[c] < comparacion.loc[c, "P10_Historico"]).mean()
    for c in variables
]

print("COMPARACION DE CARTERAS APROBADAS")
display(comparacion.round(4))

# comprobamos los incumplimientos observados frente a los esperados
print("\nDEFAULTS ENTRE LOS APROBADOS HISTORICOS")
print("Cantidad de aprobados:", len(historicos))
print("Defaults reales:", int((historicos["Estado_Final"] == 0).sum()))
print("Defaults esperados:", round(
    (1 - historicos["Prob_Pago"]).sum(), 2
))
print("Probabilidad media:", round(historicos["Prob_Pago"].mean(), 4))
print("Probabilidad media enero:", round(enero["Prob_Pago"].mean(), 4))

# estudiamos el riesgo segun la tasa de interes
cortes_tasa = np.unique(
    np.quantile(enero["Tasa_Interes"], [1/3, 2/3])
)

def grupo_tasa(datos):
    resultado = datos.copy()
    resultado["Grupo_Tasa"] = np.searchsorted(
        cortes_tasa,
        resultado["Tasa_Interes"].to_numpy(),
        side="right"
    ) + 1
    return resultado

historicos = grupo_tasa(historicos)
enero = grupo_tasa(enero)

print("\nRIESGO HISTORICO DE LOS APROBADOS POR TASA")

tabla_riesgo = historicos.groupby("Grupo_Tasa").agg(
    Aprobados=("Estado_Final", "size"),
    Defaults_Reales=("Estado_Final", lambda x: int((x == 0).sum())),
    Pago_Real=("Estado_Final", "mean"),
    Pago_Predicho=("Prob_Pago", "mean"),
    Tasa_Media=("Tasa_Interes", "mean")
)

tabla_riesgo["Defaults_Esperados"] = (
    historicos.assign(
        Default_Esperado=1 - historicos["Prob_Pago"]
    ).groupby("Grupo_Tasa")["Default_Esperado"].sum()
)

display(tabla_riesgo.round(4))

print("\nCARTERA DE ENERO POR TASA")
display(
    enero.groupby("Grupo_Tasa").agg(
        Aprobados=("ID_Solicitud", "size"),
        Capital=("Monto_Solicitado", "sum"),
        Tasa_Media=("Tasa_Interes", "mean"),
        Pago_Predicho=("Prob_Pago", "mean")
    ).round(4)
)

# comprobamos si enero contiene usuarios con solicitudes anteriores
usuarios_previos = set(train["ID_Usuario"])

coincidencias = test["ID_Usuario"].isin(usuarios_previos)

print("\nUSUARIOS COMPARTIDOS ENTRE TRAIN Y TEST")
print("Cantidad:", int(coincidencias.sum()))
print("Porcentaje:", round(100 * coincidencias.mean(), 2))


### Modelo 24 — Ajuste de probabilidad en préstamos de tasa alta

**Qué probamos.** Restamos 1,5 puntos porcentuales a la probabilidad de pago de solicitudes con tasas de interés elevadas. Después volvimos a ordenar y seleccionar los préstamos con el mismo presupuesto. Fue una prueba de sensibilidad, no una corrección aprendida de los datos.

**Qué pasó.** La diferencia promedio frente a la V7 fue de apenas USD 0,82 por mes. En diciembre perdió USD 7.412,61. Para enero habría sacado 24 préstamos e incorporado 48, sin evidencia de que ese cambio resultara mejor.

**Qué decidimos.** Descartar la penalización y conservar las probabilidades originales de la V7.

In [ ]:

import numpy as np
import pandas as pd

# probamos una estimacion mas conservadora en tasas elevadas
m24_penalizacion = 0.015
m24_corte_tasa = float(
    m22_test.loc[
        m22_test["Aprobar"] == 1,
        "Tasa_Interes"
    ].quantile(2/3)
)

def ajustar_probabilidades_m24(datos, probabilidades):
    probabilidades = np.asarray(probabilidades, dtype=float).copy()

    tasas = datos["Tasa_Interes"].to_numpy(float)
    tasas_altas = tasas >= m24_corte_tasa

    probabilidades[tasas_altas] -= m24_penalizacion

    return np.clip(probabilidades, 0.000001, 0.999999)


m24_resultados = []

# evaluamos sobre los mismos siete meses de la V7
for mes in m21_meses:
    datos = m21_predicciones[mes]["Datos"].reset_index(drop=True)
    prob_v7 = m21_predicciones[mes]["Probabilidades"]["V7"]

    prob_m24 = ajustar_probabilidades_m24(datos, prob_v7)

    presupuesto = (
        datos["Monto_Solicitado"].sum() * m19_proporcion
    )

    metricas_v7 = evaluar_cartera_m21(
        datos, prob_v7, presupuesto
    )

    metricas_m24 = evaluar_cartera_m21(
        datos, prob_m24, presupuesto
    )

    cartera_v7_mes = seleccionar_cartera_m20(
        datos, prob_v7, presupuesto
    )
    cartera_m24_mes = seleccionar_cartera_m20(
        datos, prob_m24, presupuesto
    )

    ids_v7 = set(
        cartera_v7_mes.loc[
            cartera_v7_mes["Aprobar"] == 1,
            "ID_Solicitud"
        ]
    )

    ids_m24 = set(
        cartera_m24_mes.loc[
            cartera_m24_mes["Aprobar"] == 1,
            "ID_Solicitud"
        ]
    )

    m24_resultados.append({
        "Mes": mes,
        "Ganancia_V7": metricas_v7["Ganancia"],
        "Ganancia_M24": metricas_m24["Ganancia"],
        "Diferencia": (
            metricas_m24["Ganancia"] - metricas_v7["Ganancia"]
        ),
        "Defaults_V7": metricas_v7["Defaults"],
        "Defaults_M24": metricas_m24["Defaults"],
        "Salen": len(ids_v7 - ids_m24),
        "Entran": len(ids_m24 - ids_v7)
    })

    print("Terminado:", mes)

m24_tabla = pd.DataFrame(m24_resultados)

print("\nGANANCIAS POR MES")
display(m24_tabla.round(2))

print("\nRESUMEN")
print("Ganancia promedio V7:", round(m24_tabla["Ganancia_V7"].mean(), 2))
print("Ganancia promedio M24:", round(m24_tabla["Ganancia_M24"].mean(), 2))
print("Diferencia promedio:", round(m24_tabla["Diferencia"].mean(), 2))
print("Meses ganados:", int((m24_tabla["Diferencia"] > 0).sum()), "de 7")
print("Meses sin cambios:", int((m24_tabla["Diferencia"] == 0).sum()))
print("Corte de tasa:", round(m24_corte_tasa, 4))

# comprobamos cuanto cambiaria la cartera de enero
prob_v7_enero = m22_test["Prob_Pago"].to_numpy(float)

prob_m24_enero = ajustar_probabilidades_m24(
    m22_test, prob_v7_enero
)

cartera_enero_m24 = seleccionar_cartera_m20(
    m22_test, prob_m24_enero, 2_000_000
)

ids_v7_enero = set(
    m22_test.loc[
        m22_test["Aprobar"] == 1,
        "ID_Solicitud"
    ]
)

ids_m24_enero = set(
    cartera_enero_m24.loc[
        cartera_enero_m24["Aprobar"] == 1,
        "ID_Solicitud"
    ]
)

print("\nCAMBIOS EN LA CARTERA DE ENERO")
print("Prestamos que salen:", len(ids_v7_enero - ids_m24_enero))
print("Prestamos que entran:", len(ids_m24_enero - ids_v7_enero))

aprobados_m24 = cartera_enero_m24[
    cartera_enero_m24["Aprobar"] == 1
]

print("Aprobados M24:", len(aprobados_m24))
print("Capital M24:", aprobados_m24["Monto_Solicitado"].sum())


<a id="CONCLUSION"></a>
## 9. Conclusión general

Probamos varias formas de estimar si un préstamo se iba a pagar: regresión logística, KNN, árboles de decisión, Random Forest y Gradient Boosting. También incorporamos el historial bancario, armamos variables de atrasos y comparamos distintas maneras de seleccionar solicitudes.

La Submission V7 fue la mejor que llegamos a confirmar en Kaggle: USD 130.062 en el leaderboard público, que utiliza aproximadamente el 30% del test. Entrenamos una regresión logística con `C=0.1`, características financieras y bancarias, y montos de entrenamiento dentro del rango del test. Aprobó 605 préstamos por USD 1.999.900.
